# 03 - Inference
Loads the packaged pipeline (`models/model.joblib`) exactly as the API does and scores a few patients. The second half calls the running API.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 140)

In [2]:
import json
from heartrisk.predict import RiskModel
model = RiskModel.load()
print(model.meta['model_name'], model.version, 'threshold', model.threshold)

logistic_regression 1.0.0+31a61fa threshold 0.3


In [3]:
patients = [json.load(open('samples/high_risk.json')), json.load(open('samples/low_risk.json'))]
pd.DataFrame(model.predict(patients))

,prediction,label,probability_disease,confidence,risk_band,threshold,model_version
0,1,heart disease risk,0.9932,0.9932,high,0.3,1.0.0+31a61fa
1,0,no heart disease risk,0.0771,0.9229,low,0.3,1.0.0+31a61fa


Missing `ca` / `thal` are allowed - the pipeline imputes them.

In [4]:
p = dict(patients[0], ca=None, thal=None)
model.predict([p])

[{'prediction': 1,
  'label': 'heart disease risk',
  'probability_disease': 0.926,
  'confidence': 0.926,
  'risk_band': 'high',
  'threshold': 0.3,
  'model_version': '1.0.0+31a61fa'}]

## Call the API (start it first: `uvicorn heartrisk.api:app --port 8000` or the Docker container)

In [5]:
import urllib.request
def call(url, payload=None):
    req = urllib.request.Request(url, data=json.dumps(payload).encode() if payload else None,
                                 headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())
try:
    print(call('http://localhost:8000/predict', patients[0]))
except Exception as e:
    print('API not running:', e)

{'prediction': 1, 'label': 'heart disease risk', 'probability_disease': 0.9932, 'confidence': 0.9932, 'risk_band': 'high', 'threshold': 0.3, 'model_version': '1.0.0+unknown'}
